In [6]:
# =========================
# IMPORTS
# =========================
import os
import cv2
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader

# =========================
# MOUNT DRIVE
# =========================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# =========================
# UNZIP DATASET
# =========================
import zipfile

zip_path = "/content/drive/MyDrive/AVEC2014.zip"
extract_path = "/content/avec2014"

if not os.path.exists(extract_path + "/AVEC2014"):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print("Dataset ready at:", extract_path)

# =========================
# LOAD LABELS (FIXED FOR BOTH SPLITS)
# =========================
def load_labels():
    df = pd.read_csv("/content/avec2014/AVEC2014/labels.csv")

    labels = {}
    for _, row in df.iterrows():
        key = str(row["filename"]).strip()
        key = key.replace("\\", "/")

        key = key.replace("Training/", "")
        key = key.replace("Testing/", "")

        labels[key] = float(row["BDI-II"])

    print("Labels loaded:", len(labels))
    return labels

# =========================
# VIDEO LOADER
# =========================
def load_video(path, max_frames=16):
    cap = cv2.VideoCapture(path)
    frames = []

    if not cap.isOpened():
        return None

    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (112, 112))
        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        return None

    while len(frames) < max_frames:
        frames.append(frames[-1])

    frames = np.array(frames)
    frames = np.transpose(frames, (3, 0, 1, 2))
    frames = frames / 255.0

    return torch.tensor(frames, dtype=torch.float32)

# =========================
# DATASET (WORKS FOR BOTH TRAIN + TEST)
# =========================
class AVECDataset(Dataset):
    def __init__(self, video_dir, labels):
        self.samples = []

        for root, _, files in os.walk(video_dir):
            for f in files:
                if f.endswith(".mp4"):

                    path = os.path.join(root, f)

                    # normalize key
                    rel_path = path.split("AVEC2014/")[-1]

                    if rel_path.startswith("Training/"):
                        rel_path = rel_path.replace("Training/", "")
                    elif rel_path.startswith("Testing/"):
                        rel_path = rel_path.replace("Testing/", "")

                    rel_path = os.path.splitext(rel_path)[0]
                    rel_path = rel_path.replace("\\", "/").strip()

                    if rel_path in labels:
                        self.samples.append((path, labels[rel_path]))

        print("Total usable samples:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        video = load_video(path)

        if video is None:
            return self.__getitem__((idx + 1) % len(self.samples))

        return video, torch.tensor(label, dtype=torch.float32)

# =========================
# SWITCH HERE (TRAIN OR TEST)
# =========================

# 🔴 FOR TRAINING:
# video_dir = "/content/avec2014/AVEC2014/Training"

# 🟢 FOR EVALUATION:
video_dir = "/content/avec2014/AVEC2014/Testing"

labels = load_labels()
dataset = AVECDataset(video_dir, labels)

if len(dataset) == 0:
    raise ValueError("Dataset is empty — path mismatch still exists.")

loader = DataLoader(dataset, batch_size=2, shuffle=False)

print("Pipeline ready.")

Mounted at /content/drive
Dataset ready at: /content/avec2014
Labels loaded: 300
Total usable samples: 100
Pipeline ready.


In [ ]:
# =========================
# IMPORTS
# =========================
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from google.colab import drive

# =========================
# MOUNT DRIVE
# =========================
drive.mount('/content/drive', force_remount=True)

checkpoint_path = "/content/drive/MyDrive/c3d_checkpoint_stage1.pth"
best_model_path = "/content/drive/MyDrive/best_c3d_stage1.pth"

# =========================
# DELETE OLD CHECKPOINT (IMPORTANT)
# =========================
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
    print("Old checkpoint deleted. Starting fresh Stage 1.")

# =========================
# C3D MODEL (MATCHING PAPER STRUCTURE)
# =========================
class C3D(nn.Module):
    def __init__(self):
        super(C3D, self).__init__()

        self.features = nn.Sequential(
            nn.Conv3d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.Conv3d(256, 256, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(256, 512, 3, padding=1),
            nn.ReLU(),
            nn.Conv3d(512, 512, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(512 * 1 * 7 * 7, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# =========================
# DATASET
# =========================
class AVECDataset(Dataset):
    def __init__(self, video_dir, labels):
        self.samples = []
        self.labels = labels

        for root, _, files in os.walk(video_dir):
            for f in files:
                if f.endswith(".mp4"):
                    path = os.path.join(root, f)

                    key = path.split("AVEC2014/")[-1]
                    key = os.path.splitext(key)[0]

                    if key in labels:
                        self.samples.append((path, labels[key]))

        print("Samples:", len(self.samples))

    def load_clip(self, path):
        cap = cv2.VideoCapture(path)
        frames = []

        while len(frames) < 16:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.resize(frame, (112, 112))
            frames.append(frame)

        cap.release()

        if len(frames) < 16:
            return None

        frames = np.array(frames)
        frames = np.transpose(frames, (3, 0, 1, 2)) / 255.0

        return torch.tensor(frames, dtype=torch.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        clip = self.load_clip(path)

        if clip is None:
            return self.__getitem__((idx + 1) % len(self.samples))

        return clip, torch.tensor(label, dtype=torch.float32)

# =========================
# LOAD LABELS
# =========================
def load_labels():
    df = pd.read_csv("/content/avec2014/AVEC2014/labels.csv")

    labels = {}
    for _, row in df.iterrows():
        key = str(row["filename"]).strip().replace("\\", "/")
        labels[key] = float(row["BDI-II"])

    return labels

# =========================
# SETUP
# =========================
video_dir = "/content/avec2014/AVEC2014/Training"
labels = load_labels()

dataset = AVECDataset(video_dir, labels)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = C3D().to(device)

# =========================
# STAGE 1 FREEZING
# =========================
for param in model.features.parameters():
    param.requires_grad = False

print("Stage 1: Feature extractor frozen.")

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5
)

# =========================
# TRAINING WITH EARLY STOPPING
# =========================
def train():

    model.train()

    best_loss = float("inf")
    patience = 3
    counter = 0

    for epoch in range(20):

        total_loss = 0

        for videos, targets in loader:
            videos = videos.to(device)
            targets = targets.to(device).view(-1, 1)

            preds = model(videos)
            loss = criterion(preds, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")

        # SAVE CHECKPOINT
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "loss": avg_loss
        }, checkpoint_path)

        # EARLY STOPPING
        if avg_loss < best_loss:
            best_loss = avg_loss
            counter = 0

            torch.save(model.state_dict(), best_model_path)
            print("Best model saved.")

        else:
            counter += 1
            print(f"No improvement ({counter}/{patience})")

        if counter >= patience:
            print("Early stopping triggered.")
            break

# =========================
# START TRAINING
# =========================
train()

Mounted at /content/drive
Samples: 100
Stage 1: Feature extractor frozen.
Epoch 1 | Loss: 341.7907
Best model saved.
Epoch 2 | Loss: 182.0180
Best model saved.
Epoch 3 | Loss: 146.2767
Best model saved.
Epoch 4 | Loss: 142.5611
Best model saved.
Epoch 5 | Loss: 147.1927
No improvement (1/3)
Epoch 6 | Loss: 149.0640
No improvement (2/3)
Epoch 7 | Loss: 146.4786
No improvement (3/3)
Early stopping triggered.


In [ ]:
# =========================
# STAGE 2: FINE-TUNING (UNFREEZE)
# =========================

print("Starting Stage 2 fine-tuning...")

# Load best Stage 1 model
model.load_state_dict(torch.load(best_model_path, map_location=device))

# =========================
# UNFREEZE ALL LAYERS (or partial unfreeze)
# =========================
for param in model.features.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

print("All layers unfrozen for fine-tuning.")

# =========================
# LOWER LEARNING RATE
# =========================
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

criterion = nn.MSELoss()

# =========================
# OPTIONAL: resume training state reset
# =========================
start_epoch = 0

# =========================
# TRAINING LOOP (STAGE 2)
# =========================
def train_stage2():

    model.train()

    best_loss = float("inf")
    patience = 5
    counter = 0

    for epoch in range(20):

        total_loss = 0

        for videos, targets in loader:
            videos = videos.to(device)
            targets = targets.to(device).view(-1, 1)

            preds = model(videos)
            loss = criterion(preds, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)

        print(f"[Stage 2] Epoch {epoch+1} | Loss: {avg_loss:.4f}")

        # save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            counter = 0

            torch.save(model.state_dict(),
                       "/content/drive/MyDrive/best_c3d_stage2.pth")

            print("Stage 2 best model saved.")

        else:
            counter += 1
            print(f"No improvement ({counter}/{patience})")

        if counter >= patience:
            print("Stage 2 early stopping triggered.")
            break

# =========================
# RUN STAGE 2
# =========================
train_stage2()

Starting Stage 2 fine-tuning...
All layers unfrozen for fine-tuning.
[Stage 2] Epoch 1 | Loss: 143.1912
Stage 2 best model saved.
[Stage 2] Epoch 2 | Loss: 142.5398
Stage 2 best model saved.
[Stage 2] Epoch 3 | Loss: 148.8807
No improvement (1/5)
[Stage 2] Epoch 4 | Loss: 139.1817
Stage 2 best model saved.
[Stage 2] Epoch 5 | Loss: 143.7769
No improvement (1/5)
[Stage 2] Epoch 6 | Loss: 130.4662
Stage 2 best model saved.
[Stage 2] Epoch 7 | Loss: 128.5019
Stage 2 best model saved.
[Stage 2] Epoch 8 | Loss: 117.6947
Stage 2 best model saved.
[Stage 2] Epoch 9 | Loss: 122.4963
No improvement (1/5)
[Stage 2] Epoch 10 | Loss: 112.2194
Stage 2 best model saved.
[Stage 2] Epoch 11 | Loss: 99.9210
Stage 2 best model saved.
[Stage 2] Epoch 12 | Loss: 85.5715
Stage 2 best model saved.
[Stage 2] Epoch 13 | Loss: 94.3089
No improvement (1/5)
[Stage 2] Epoch 14 | Loss: 82.7574
Stage 2 best model saved.
[Stage 2] Epoch 15 | Loss: 67.2781
Stage 2 best model saved.
[Stage 2] Epoch 16 | Loss: 79.0140


In [7]:
# =========================
# RECREATE MODEL
# =========================
import torch.nn as nn

class C3D(nn.Module):
    def __init__(self):
        super(C3D, self).__init__()

        self.features = nn.Sequential(
            nn.Conv3d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.Conv3d(256, 256, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(256, 512, 3, padding=1),
            nn.ReLU(),
            nn.Conv3d(512, 512, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(512 * 1 * 7 * 7, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


# =========================
# LOAD MODEL
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = C3D().to(device)

# load saved weights
model.load_state_dict(torch.load("/content/drive/MyDrive/emergency_backup_c3d.pth", map_location=device))

model.eval()

print("Model loaded successfully.")

Model loaded successfully.


In [8]:
# =========================
# IMPORTS
# =========================
import numpy as np
import torch

# =========================
# MODEL (must already exist in memory OR redefine if needed)
# =========================
model.eval()

# =========================
# MULTI-CLIP FUNCTION
# =========================
def get_clips(video_tensor, clip_len=16, stride=8):
    clips = []
    T = video_tensor.shape[1]

    if T < clip_len:
        pad = clip_len - T
        last_frame = video_tensor[:, -1:, :, :]
        padding = last_frame.repeat(1, pad, 1, 1)
        video_tensor = torch.cat([video_tensor, padding], dim=1)
        T = clip_len

    for i in range(0, T - clip_len + 1, stride):
        clip = video_tensor[:, i:i+clip_len, :, :]
        clips.append(clip)

    return clips
# =========================
# PREDICTION FUNCTION
# =========================
def predict_video(video_tensor):
    clips = get_clips(video_tensor)

    preds = []

    with torch.no_grad():
        for clip in clips:
            clip = clip.unsqueeze(0).to(device)
            pred = model(clip)
            preds.append(pred.item())

    if len(preds) == 0:
        return None

    return np.mean(preds)

# =========================
# RUN EVALUATION
# =========================
y_true = []
y_pred = []

for video, label in dataset:

    pred = predict_video(video)

    if pred is not None:
        y_true.append(label)
        y_pred.append(pred)

# =========================
# SAFETY CHECK
# =========================
print("Valid samples used:", len(y_true))

if len(y_true) == 0:
    print("ERROR: No valid predictions. Check dataset/video loading.")
else:

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # =========================
    # MAE
    # =========================
    mae = np.mean(np.abs(y_true - y_pred))

    # =========================
    # RMSE
    # =========================
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    # =========================
    # CCC (Concordance Correlation Coefficient)
    # =========================
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)

    var_true = np.var(y_true)
    var_pred = np.var(y_pred)

    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))

    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2 + 1e-8)

    # =========================
    # RESULTS
    # =========================
    print("\n===== FINAL PAPER-STYLE RESULTS =====")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"CCC  : {ccc:.4f}")

Valid samples used: 100

===== FINAL PAPER-STYLE RESULTS =====
MAE  : 8.6473
RMSE : 10.6035
CCC  : 0.4141


In [ ]:
import torch

torch.save(model.state_dict(),
           "/content/drive/MyDrive/emergency_backup_c3d.pth")

print("Backup saved safely.")

Backup saved safely.


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_csv("/content/avec2014/AVEC2014/labels.csv")
print(df.columns)
print(df.head())

104274


In [ ]:
print(find_videos("/content/avec2014/AVEC2014")[0])
print(list(labels.keys())[0])